### ML Code
Time to make some models! 


In [19]:
# First we got to do some imports (I'm just grabbing the ones from 02 for now)
from pathlib import Path
import platform
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
mae = mean_absolute_error
mape = mean_absolute_percentage_error

# Random seed for reproducibility
np.random.seed(42)

df = pd.read_csv("data/ecowas_df_synthetic_full.csv")

In [52]:
df.columns

Index(['Unnamed: 0', 'country', 'year', 'disorder_Demonstrations',
       'disorder_Political violence',
       'disorder_Political violence; Demonstrations',
       'disorder_Strategic developments', 'event_Battles',
       'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
       'event_Strategic developments', 'event_Violence against civilians',
       'perpetrator_Civilians', 'perpetrator_External/Other forces',
       'perpetrator_Identity militia', 'perpetrator_Political militia',
       'perpetrator_Protesters', 'perpetrator_Rebel group',
       'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
       'target_External/Other forces', 'target_Identity militia',
       'target_Political militia', 'target_Protesters', 'target_Rebel group',
       'target_Rioters', 'target_State forces', 'fatalities', 'iso3_o',
       'iso3_d', 'distw_harmonic', 'distw_arithmetic', 'dist', 'distcap',
       'contig', 'diplo_disagreement', 'comlang_off', 'comlang

## Feature Engineering: Log1p Transform + Rolling Conflict Aggregates

**Step 1** — `log(1 + x)` on all conflict count columns. Same motivation as log-trade: conflict counts are right-skewed, so proportional changes matter more than absolute ones.

**Step 6** — 3-year rolling mean per exporter. Captures cumulative conflict pressure (J-curve effect: trade recovers gradually, so recent history matters).

In [ ]:
conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)] + ["fatalities"]

df_log = df.copy()

# Step 1: log1p transform on all conflict columns
for col in conflict_cols:
    df_log[col] = np.log1p(df_log[col])

# Step 6: 3-year rolling mean per exporter (sorted to ensure chronological order)
df_log = df_log.sort_values(["iso3_o", "year"]).reset_index(drop=True)
roll_cols = []
for col in conflict_cols:
    roll_col = f"{col}_roll3"
    df_log[roll_col] = (
        df_log.groupby("iso3_o")[col]
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )
    roll_cols.append(roll_col)

print(f"df_log ready — {len(conflict_cols)} conflict cols log1p-transformed, {len(roll_cols)} _roll3 cols added")
print(f"Total columns in df_log: {len(df_log.columns)}")

We want to make XGBoost, Random Forest and Neural Networks

In [18]:
from xgboost import XGBRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error, mean_absolute_percentage_error
from sklearn.pipeline import Pipeline



legal_target_vars = {"tradeflow_baci", 'tradeflow_comtrade_o', 'tradeflow_comtrade_d', 'tradeflow_imf_o', 'tradeflow_imf_d', "combined_trade_baci"}
required_columns = {"gdp_o", "gdp_d", "distw_arithmetic", "comlang_ethno", "comlang_off", "contig"}


## XGBoost implementation

In [ ]:
def xgboost_regression(
    df: pd.DataFrame,
    target_variable: str,
    columns: list,
    year_split: int = 2016,
    set_random_state: int = 42,
    n_estimators: int = 500,
    learning_rate: float = 0.05,
    max_depth: int = 6,
    subsample: float = 0.8,
    colsample_bytree: float = 0.8,
    min_child_weight: int = 1,
):
    # Convert to list(dict.fromkeys()) to remove duplicates while preserving order
    columns = list(dict.fromkeys(columns)) 
    
    columns = columns.copy()

    # -----------------------------
    # Validation checks
    # -----------------------------
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}.")

    if target_variable not in legal_target_vars:
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")

    # -----------------------------
    # Train / test split
    # -----------------------------
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df  = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df  = test_df.dropna()

    # -----------------------------
    # Log transformations
    # -----------------------------
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i + 1])

    # --- Add exporter and importer fixed effects ---
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float)
        dummies_test = pd.get_dummies(test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float)
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)
        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    # -----------------------------
    # Final model matrices
    # -----------------------------
    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]
    X_test  = test_df_model[columns]
    y_test  = test_df_model[f"{target_variable}_log_trade"]

    # -----------------------------
    # XGBoost model
    # -----------------------------
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        random_state=set_random_state,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"XGBoost Out-of-sample RMSE: {rmse}")
    print(f"XGBoost Out-of-sample MAE:  {mae}")
    print(f"XGBoost Out-of-sample R²:   {r2}")

    return {
        "model_name": f"XGBoost_{target_variable}",
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }

In [68]:
XG_nocon = xgboost_regression(df, "tradeflow_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic",
       'contig', 'diplo_disagreement', 'comlang_off', 'comlang_ethno'])

XGBoost Out-of-sample RMSE: 1.5269180816263983
XGBoost Out-of-sample MAE:  1.159714325501271
XGBoost Out-of-sample R²:   0.44656473200916424


In [ ]:
# xgboost_country_year_fe was superseded — country FE now handled via iso3_o/iso3_d dummies inside xgboost_regression

In [26]:
import contextlib, io
from itertools import combinations

def find_best_conflict_combo1(df, target_variable, base_columns=None, year_split=2016):
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)] + ["fatalities"]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    results = []

    # Build all combos to test
    combos_to_test = []

    # 1) All non-empty subsets of the 4 base groups (15 combos)
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))

    # 2) All 4 groups + fatalities
    all_cols = [c for g in base_groups for c in conflict_groups[g]] + ["fatalities"]
    combos_to_test.append(("disorder + event + perpetrator + target + fatalities", all_cols))

    # 3) Fatalities alone
    combos_to_test.append(("fatalities", ["fatalities"]))

    # 4) Total conflicts alone
    combos_to_test.append(("total_conflicts", ["total_conflicts"]))

    # 5) Total conflicts + fatalities
    combos_to_test.append(("total_conflicts + fatalities", ["total_conflicts", "fatalities"]))


    # Run all combos
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = xgboost_regression(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)

    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df


In [ ]:
xgb_con = find_best_conflict_combo1(df_log, "tradeflow_baci")

## Random Forest implementation

In [ ]:
def random_forest_regression(
    df: pd.DataFrame,
    target_variable: str,
    columns: list,
    year_split: int = 2016,
    set_random_state: int = 42,
    n_estimators: int = 500,
    max_depth: int | None = None,
    min_samples_leaf: int = 1,
    min_samples_split: int = 2,
    max_features: str | float = "sqrt",
):
    """
    Random Forest regression following the conventions and transformations of the baseline linear FE model.
    """

    columns = columns.copy()

    # -----------------------------
    # Validation checks
    # -----------------------------
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}.")

    if target_variable not in legal_target_vars:
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")

    # -----------------------------
    # Train / test split
    # -----------------------------
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df  = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df  = test_df.dropna()

    # -----------------------------
    # Log transformations
    # -----------------------------
    # Log-transform target
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    # Log-transform gravity variables
    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    # Replace raw gravity vars with log versions in columns list
    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i + 1])

    # --- Add exporter and importer fixed effects ---
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(
            train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = pd.get_dummies(
            test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        # Align test columns to train (handles missing categories)
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)

        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    # -----------------------------
    # Final model matrices
    # -----------------------------
    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]

    # -----------------------------
    # Random Forest model
    # -----------------------------
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        max_features=max_features,
        random_state=set_random_state,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # -----------------------------
    # Evaluation
    # -----------------------------
    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"Random Forest Out-of-sample RMSE: {rmse}")
    print(f"Random Forest Out-of-sample MAE:  {mae}")
    print(f"Random Forest Out-of-sample R²:   {r2}")

    return {
        "model_name": f"RandomForest_{target_variable}",
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }


In [45]:
RFR_nocon = random_forest_regression(df, "tradeflow_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])

Random Forest Out-of-sample RMSE: 1.6334196317598286
Random Forest Out-of-sample MAE:  1.245933388513989
Random Forest Out-of-sample R²:   0.3285739831176768


In [30]:
import contextlib, io
from itertools import combinations

def find_best_conflict_combo2(df, target_variable, base_columns=None, year_split=2016):
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)] + ["fatalities"]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    results = []

    # Build all combos to test
    combos_to_test = []

    # 1) All non-empty subsets of the 4 base groups (15 combos)
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))

    # 2) All 4 groups + fatalities
    all_cols = [c for g in base_groups for c in conflict_groups[g]] + ["fatalities"]
    combos_to_test.append(("disorder + event + perpetrator + target + fatalities", all_cols))

    # 3) Fatalities alone
    combos_to_test.append(("fatalities", ["fatalities"]))

    # 4) Total conflicts alone
    combos_to_test.append(("total_conflicts", ["total_conflicts"]))

    # 5) Total conflicts + fatalities
    combos_to_test.append(("total_conflicts + fatalities", ["total_conflicts", "fatalities"]))


    # Run all combos
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = random_forest_regression(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)

    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df


In [ ]:
RFR_con = find_best_conflict_combo2(df_log, "tradeflow_baci")

## Neural Network implementation

In [ ]:
def multilayer_perceptron(
    df: pd.DataFrame, 
    target_variable: str, 
    columns: list, 
    year_split = 2016, 
    hidden_layer=(100, 50), 
    set_random_state=42,
    alpha=0.0001,
    learning_rate_init=0.001,
):
    """
    Multilayer Perceptron regression following the conventions and transformations of the baseline models.
    """
    columns = columns.copy()

    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}.")
    
    if target_variable not in legal_target_vars:    
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")
    
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()
    
    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df  = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df  = test_df.dropna()

    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i+1])
    
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(
            train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = pd.get_dummies(
            test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)

        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]
    X_test  = test_df_model[columns]
    y_test  = test_df_model[f"{target_variable}_log_trade"]

    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()), 
            ("mlp", MLPRegressor(
                hidden_layer_sizes=hidden_layer, 
                activation="relu",  
                solver="adam", 
                alpha=alpha,
                learning_rate_init=learning_rate_init,
                max_iter=1000, 
                random_state=set_random_state
            )),
        ]
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"MLP Out-of-sample RMSE: {rmse}")
    print(f"MLP Out-of-sample MAE:  {mae}")
    print(f"MLP Out-of-sample R²:   {r2}")

    return {
        "model_name": f"MLP_{target_variable}",
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }

In [ ]:
mlp_nocon = multilayer_perceptron(df_log, "tradeflow_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])

## Step 2: MLP Ablation Study

Mirrors `find_best_conflict_combo1` (XGBoost) and `find_best_conflict_combo2` (RF) — completes the ablation picture across all three model types.

In [ ]:
def find_best_conflict_combo3(df, target_variable, base_columns=None, year_split=2016):
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)] + ["fatalities"]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    combos_to_test = []

    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))

    all_cols = [c for g in base_groups for c in conflict_groups[g]] + ["fatalities"]
    combos_to_test.append(("disorder + event + perpetrator + target + fatalities", all_cols))
    combos_to_test.append(("fatalities", ["fatalities"]))
    combos_to_test.append(("total_conflicts", ["total_conflicts"]))
    combos_to_test.append(("total_conflicts + fatalities", ["total_conflicts", "fatalities"]))

    results = []
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = multilayer_perceptron(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)
    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — MLP {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df

In [ ]:
mlp_con = find_best_conflict_combo3(df_log, "tradeflow_baci")

## Step 3: Null Model (Permutation Test)

Shuffles conflict columns to destroy any dyad-year signal. If the real model consistently beats the null distribution, conflict variables carry genuine predictive power — not just noise. Directly answers the supervisor's request for a permutation baseline.

In [ ]:
def null_model_test(df, model_fn, target_variable, base_columns, conflict_cols,
                    n_shuffles=50, year_split=2016, random_state=42):
    """
    Permutation test: shuffles conflict columns independently across all rows,
    breaking the dyad-year linkage. Measures the null distribution of model performance
    and compares it to the real model's score.
    """
    rng = np.random.default_rng(random_state)
    null_maes, null_r2s = [], []
    all_cols = base_columns + conflict_cols

    for i in range(n_shuffles):
        df_shuffled = df.copy()
        for col in conflict_cols:
            df_shuffled[col] = rng.permutation(df_shuffled[col].values)
        with contextlib.redirect_stdout(io.StringIO()):
            res = model_fn(df_shuffled, target_variable, all_cols, year_split)
        null_maes.append(res["mae"])
        null_r2s.append(res["r2"])

    with contextlib.redirect_stdout(io.StringIO()):
        real_res = model_fn(df, target_variable, all_cols, year_split)
    real_mae = real_res["mae"]
    real_r2  = real_res["r2"]

    p_val_mae = np.mean(np.array(null_maes) <= real_mae)
    p_val_r2  = np.mean(np.array(null_r2s) >= real_r2)

    print(f"Real model  —  MAE: {real_mae:.4f}  |  R²: {real_r2:.4f}")
    print(f"Null (mean) —  MAE: {np.mean(null_maes):.4f}  |  R²: {np.mean(null_r2s):.4f}")
    print(f"p-value (MAE, lower is better): {p_val_mae:.3f}")
    print(f"p-value (R², higher is better): {p_val_r2:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(null_maes, bins=20, color="steelblue", alpha=0.7, edgecolor="white")
    axes[0].axvline(real_mae, color="crimson", linewidth=2,
                    label=f"Real model MAE = {real_mae:.4f}")
    axes[0].set_xlabel("MAE (log-trade units)")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"Null Distribution — MAE  (n={n_shuffles} shuffles)")
    axes[0].legend()

    axes[1].hist(null_r2s, bins=20, color="steelblue", alpha=0.7, edgecolor="white")
    axes[1].axvline(real_r2, color="crimson", linewidth=2,
                    label=f"Real model R² = {real_r2:.4f}")
    axes[1].set_xlabel("R²")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Null Distribution — R²  (n={n_shuffles} shuffles)")
    axes[1].legend()

    feature_label = ", ".join(conflict_cols[:3]) + ("..." if len(conflict_cols) > 3 else "")
    plt.suptitle(f"Null Model Test — conflict features: [{feature_label}]", fontsize=11)
    plt.tight_layout()
    plt.show()

    return {
        "real_mae": real_mae, "real_r2": real_r2,
        "null_mae_mean": np.mean(null_maes), "null_r2_mean": np.mean(null_r2s),
        "p_val_mae": p_val_mae, "p_val_r2": p_val_r2,
    }

In [ ]:
base_cols = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

null_results = null_model_test(
    df_log, xgboost_regression, "tradeflow_baci",
    base_columns=base_cols,
    conflict_cols=["fatalities"],
    n_shuffles=50,
)

## Step 4: Feature Importance (Permutation Importance)

Uses `sklearn.inspection.permutation_importance` — model-agnostic, no extra installs needed. Directly answers the 23/04 question: *which conflict types explain trade most, and what are their relative weights?*

In [ ]:
from sklearn.inspection import permutation_importance as sk_perm_importance

def plot_permutation_importance(df, target_variable, columns, year_split=2016,
                                 n_repeats=20, random_state=42, top_n=20):
    """
    Fits XGBoost and plots permutation importance for all features.
    Each bar shows how much R² drops when that feature is randomly shuffled.
    """
    columns = list(dict.fromkeys(columns))

    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].dropna().copy()
    test_df  = test_df_full[all_needed].dropna().copy()

    train_df["log_trade"] = np.log(train_df[target_variable])
    test_df["log_trade"]  = np.log(test_df[target_variable])

    col_map = {"gdp_o": "log_gdp_o", "gdp_d": "log_gdp_d", "distw_arithmetic": "log_distw"}
    for raw, logged in col_map.items():
        if raw in columns:
            train_df[logged] = np.log(train_df[raw])
            test_df[logged]  = np.log(test_df[raw])

    feat_cols = [col_map.get(c, c) for c in columns]

    X_train = train_df[feat_cols]
    y_train = train_df["log_trade"]
    X_test  = test_df[feat_cols]
    y_test  = test_df["log_trade"]

    model = XGBRegressor(
        objective="reg:squarederror", n_estimators=500, learning_rate=0.05,
        max_depth=6, subsample=0.8, colsample_bytree=0.8,
        random_state=random_state, n_jobs=-1,
    )
    model.fit(X_train, y_train)

    perm = sk_perm_importance(model, X_test, y_test, n_repeats=n_repeats,
                               random_state=random_state, n_jobs=-1)

    imp_df = pd.DataFrame({
        "feature":          feat_cols,
        "importance_mean":  perm.importances_mean,
        "importance_std":   perm.importances_std,
    }).sort_values("importance_mean", ascending=False).head(top_n)

    imp_df_plot = imp_df.sort_values("importance_mean", ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(5, len(imp_df_plot) * 0.4)))
    ax.barh(imp_df_plot["feature"], imp_df_plot["importance_mean"],
            xerr=imp_df_plot["importance_std"], color="steelblue", alpha=0.8,
            capsize=3, edgecolor="white")
    ax.set_xlabel("Mean decrease in R² when feature is permuted")
    ax.set_title(f"Permutation Importance — XGBoost on {target_variable}  (top {top_n})")
    ax.axvline(0, color="black", linewidth=0.8)
    plt.tight_layout()
    plt.show()

    return imp_df

In [ ]:
all_conflict_cols = [c for c in df_log.columns
                     if any(c.startswith(p) for p in conflict_prefixes)] + ["fatalities"]
base_cols = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

imp_df = plot_permutation_importance(
    df_log, "tradeflow_baci",
    columns=base_cols + all_conflict_cols,
    top_n=20,
)
print(imp_df.to_string(index=False))

## Step 5: XGBoost Hyperparameter Tuning

`TimeSeriesSplit` respects temporal ordering — no future leakage into CV folds. `RandomizedSearchCV` with `n_iter=30` searches the param space efficiently without exhaustive grid search. Run this cell once; the best params can then be passed directly to `xgboost_regression`.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

def tune_xgboost(df, target_variable, columns, year_split=2016, n_iter=30, random_state=42):
    """
    Finds optimal XGBoost hyperparameters using TimeSeriesSplit CV.
    Only trains on the training split (year <= year_split) to avoid leakage.
    """
    columns = list(dict.fromkeys(columns))

    train_df_full = df[df["year"] <= year_split].copy()
    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].dropna().copy()

    train_df["log_trade"] = np.log(train_df[target_variable])
    col_map = {"gdp_o": "log_gdp_o", "gdp_d": "log_gdp_d", "distw_arithmetic": "log_distw"}
    for raw, logged in col_map.items():
        if raw in columns:
            train_df[logged] = np.log(train_df[raw])
    feat_cols = [col_map.get(c, c) for c in columns]

    X = train_df[feat_cols]
    y = train_df["log_trade"]

    param_dist = {
        "n_estimators":     [200, 500, 1000],
        "max_depth":        [3, 4, 5, 6, 7],
        "learning_rate":    [0.01, 0.05, 0.1],
        "subsample":        [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "min_child_weight": [1, 3, 5],
    }

    tscv = TimeSeriesSplit(n_splits=5)
    base_model = XGBRegressor(objective="reg:squarederror", random_state=random_state, n_jobs=-1)

    search = RandomizedSearchCV(
        base_model, param_dist, n_iter=n_iter, cv=tscv,
        scoring="neg_mean_absolute_error", n_jobs=-1,
        random_state=random_state, verbose=1,
    )
    search.fit(X, y)

    print(f"\nBest CV MAE: {-search.best_score_:.4f}")
    print(f"Best params: {search.best_params_}")
    return search.best_params_

best_params = tune_xgboost(
    df_log, "tradeflow_baci",
    columns=base_cols + ["fatalities"],
)

# Re-run XGBoost with tuned params
XG_tuned = xgboost_regression(
    df_log, "tradeflow_baci",
    columns=base_cols + ["fatalities"],
    **best_params,
)

### Random Forest Hyperparameter Tuning

In [ ]:
def tune_random_forest(df, target_variable, columns, year_split=2016, n_iter=30, random_state=42):
    """
    Finds optimal Random Forest hyperparameters using TimeSeriesSplit CV.
    Only trains on the training split (year <= year_split) to avoid leakage.
    """
    columns = list(dict.fromkeys(columns))

    train_df_full = df[df["year"] <= year_split].copy()
    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].dropna().copy()

    train_df["log_trade"] = np.log(train_df[target_variable])
    col_map = {"gdp_o": "log_gdp_o", "gdp_d": "log_gdp_d", "distw_arithmetic": "log_distw"}
    for raw, logged in col_map.items():
        if raw in columns:
            train_df[logged] = np.log(train_df[raw])
    feat_cols = [col_map.get(c, c) for c in columns]

    X = train_df[feat_cols]
    y = train_df["log_trade"]

    param_dist = {
        "n_estimators":      [200, 500, 1000],
        "max_depth":         [None, 10, 20, 30],
        "min_samples_leaf":  [1, 2, 4, 8],
        "min_samples_split": [2, 5, 10],
        "max_features":      ["sqrt", "log2", 0.5],
    }

    tscv = TimeSeriesSplit(n_splits=5)
    base_model = RandomForestRegressor(random_state=random_state, n_jobs=-1)

    search = RandomizedSearchCV(
        base_model, param_dist, n_iter=n_iter, cv=tscv,
        scoring="neg_mean_absolute_error", n_jobs=-1,
        random_state=random_state, verbose=1,
    )
    search.fit(X, y)

    print(f"\nBest CV MAE: {-search.best_score_:.4f}")
    print(f"Best params: {search.best_params_}")
    return search.best_params_

best_rf_params = tune_random_forest(
    df_log, "tradeflow_baci",
    columns=base_cols + ["fatalities"],
)

RFR_tuned = random_forest_regression(
    df_log, "tradeflow_baci",
    columns=base_cols + ["fatalities"],
    **best_rf_params,
)

### MLP Hyperparameter Tuning

In [ ]:
def tune_mlp(df, target_variable, columns, year_split=2016, n_iter=30, random_state=42):
    """
    Finds optimal MLP hyperparameters using TimeSeriesSplit CV.
    Uses a Pipeline with StandardScaler (required for MLP) and mlp__ prefixed params.
    Returns params ready to unpack into multilayer_perceptron().
    """
    columns = list(dict.fromkeys(columns))

    train_df_full = df[df["year"] <= year_split].copy()
    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].dropna().copy()

    train_df["log_trade"] = np.log(train_df[target_variable])
    col_map = {"gdp_o": "log_gdp_o", "gdp_d": "log_gdp_d", "distw_arithmetic": "log_distw"}
    for raw, logged in col_map.items():
        if raw in columns:
            train_df[logged] = np.log(train_df[raw])
    feat_cols = [col_map.get(c, c) for c in columns]

    X = train_df[feat_cols]
    y = train_df["log_trade"]

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            activation="relu", solver="adam", max_iter=1000, random_state=random_state
        )),
    ])

    param_dist = {
        "mlp__hidden_layer_sizes": [(50,), (100,), (100, 50), (200, 100), (100, 50, 25)],
        "mlp__alpha":              [0.0001, 0.001, 0.01],
        "mlp__learning_rate_init": [0.0001, 0.001, 0.01],
    }

    tscv = TimeSeriesSplit(n_splits=5)

    search = RandomizedSearchCV(
        pipeline, param_dist, n_iter=n_iter, cv=tscv,
        scoring="neg_mean_absolute_error", n_jobs=-1,
        random_state=random_state, verbose=1,
    )
    search.fit(X, y)

    # Strip "mlp__" prefix so params map to multilayer_perceptron() kwargs
    best = {k.replace("mlp__hidden_layer_sizes", "hidden_layer")
             .replace("mlp__", ""): v
            for k, v in search.best_params_.items()}

    print(f"\nBest CV MAE: {-search.best_score_:.4f}")
    print(f"Best params: {best}")
    return best

best_mlp_params = tune_mlp(
    df_log, "tradeflow_baci",
    columns=base_cols + ["fatalities"],
)

MLP_tuned = multilayer_perceptron(
    df_log, "tradeflow_baci",
    columns=base_cols + ["fatalities"],
    **best_mlp_params,
)

## Step 8: Same Models on `combined_trade_baci` — Apples-to-Apples vs OLS FE

The OLS FE baseline (R²=0.5614) was run on `combined_trade_baci`, an aggregate of BACI + IMF + Comtrade flows that is denser and less noisy than `tradeflow_baci` alone. To make the ML-vs-linear comparison honest, we re-run all three ML models on the *same* target. If ML still beats OLS FE here, the win is real.

In [ ]:
### XGBoost on combined_trade_baci ###
print("="*100)
print("  XGBoost — no conflict (gravity baseline only)")
print("="*100)
XG_nocon_ct = xgboost_regression(
    df_log, "combined_trade_baci",
    columns=base_cols,
)

print("\n" + "="*100)
print("  XGBoost — full conflict ablation")
print("="*100)
xgb_con_ct = find_best_conflict_combo1(df_log, "combined_trade_baci")

In [ ]:
### Random Forest on combined_trade_baci ###
print("="*100)
print("  Random Forest — no conflict (gravity baseline only)")
print("="*100)
RFR_nocon_ct = random_forest_regression(
    df_log, "combined_trade_baci",
    columns=base_cols,
)

print("\n" + "="*100)
print("  Random Forest — full conflict ablation")
print("="*100)
RFR_con_ct = find_best_conflict_combo2(df_log, "combined_trade_baci")

In [ ]:
### MLP on combined_trade_baci ###
print("="*100)
print("  MLP — no conflict (gravity baseline only)")
print("="*100)
mlp_nocon_ct = multilayer_perceptron(
    df_log, "combined_trade_baci",
    columns=base_cols,
)

print("\n" + "="*100)
print("  MLP — full conflict ablation")
print("="*100)
mlp_con_ct = find_best_conflict_combo3(df_log, "combined_trade_baci")

## Step 7: Model Comparison Table

Thesis-ready summary. Linear baselines come from `04_modelling copy.ipynb`:

- **OLS Gravity (no FE)** on `tradeflow_baci` — log scale
- **OLS Linear (no FE) + best conflict combo** on `tradeflow_baci` — log scale
- **OLS Linear FE + best conflict** (`fatalities`) on `combined_trade_baci` — log scale, different target
- **PPML FE (no conflict)** on `combined_trade_baci` — **levels** scale (raw $, not log)
- **PPML FE + best conflict by MAE** (`disorder + event + target`) on `tradeflow_baci` — **levels** scale

The `Target / Scale` column flags the comparison caveat: PPML rows are not directly comparable to log-scale rows in absolute RMSE/MAE, but R² is dimensionless.

In [ ]:
rows = [
    # ---- Linear / statistical baselines from 04_modelling copy.ipynb ----
    {"Model": "OLS Gravity Baseline",   "Conflict Features": "none",
     "Target / Scale": "tradeflow_baci (log)",
     "N Train": 3458, "N Test": 910,
     "RMSE": 1.6233, "MAE": float("nan"), "R²": 0.3369},
    {"Model": "OLS Linear (no FE)",     "Conflict Features": "total_conflicts + fatalities",
     "Target / Scale": "tradeflow_baci (log)",
     "N Train": 3458, "N Test": 910,
     "RMSE": 1.6235, "MAE": 1.2787, "R²": 0.3367},
    {"Model": "OLS Linear FE",          "Conflict Features": "fatalities (best)",
     "Target / Scale": "combined_trade_baci (log)",
     "N Train": 3458, "N Test": 910,
     "RMSE": 1.0736, "MAE": 0.8259, "R²": 0.5614},
    {"Model": "PPML FE (gravity only)", "Conflict Features": "none",
     "Target / Scale": "combined_trade_baci (levels)",
     "N Train": 3458, "N Test": 910,
     "RMSE": 264197.5222, "MAE": 110332.7129, "R²": 0.2857},
    {"Model": "PPML FE",                "Conflict Features": "disorder + event + target (best MAE)",
     "Target / Scale": "tradeflow_baci (levels)",
     "N Train": 3458, "N Test": 910,
     "RMSE": 182090.5384, "MAE": 61531.6812, "R²": 0.1745},
]

def _add(label, conflict_feat, res, target_scale="tradeflow_baci (log)"):
    rows.append({
        "Model": label,
        "Conflict Features": conflict_feat,
        "Target / Scale": target_scale,
        "N Train": res["n_train"],
        "N Test":  res["n_test"],
        "RMSE":    round(res["rmse"], 4),
        "MAE":     round(res["mae"],  4),
        "R²":      round(res["r2"],   4),
    })

# ---- ML models on tradeflow_baci ----
_add("XGBoost",              "none",                           XG_nocon)
_add("XGBoost",              xgb_con.iloc[0]["model_name"],    xgb_con.iloc[0])
_add("XGBoost (tuned)",      "fatalities",                     XG_tuned)
_add("Random Forest",        "none",                           RFR_nocon)
_add("Random Forest",        RFR_con.iloc[0]["model_name"],    RFR_con.iloc[0])
_add("Random Forest (tuned)","fatalities",                     RFR_tuned)
_add("MLP",                  "none",                           mlp_nocon)
_add("MLP",                  mlp_con.iloc[0]["model_name"],    mlp_con.iloc[0])
_add("MLP (tuned)",          "fatalities",                     MLP_tuned)

# ---- ML models on combined_trade_baci (apples-to-apples vs OLS FE) ----
ct_scale = "combined_trade_baci (log)"
_add("XGBoost",       "none",                              XG_nocon_ct,  ct_scale)
_add("XGBoost",       xgb_con_ct.iloc[0]["model_name"],    xgb_con_ct.iloc[0],  ct_scale)
_add("Random Forest", "none",                              RFR_nocon_ct, ct_scale)
_add("Random Forest", RFR_con_ct.iloc[0]["model_name"],    RFR_con_ct.iloc[0],  ct_scale)
_add("MLP",           "none",                              mlp_nocon_ct, ct_scale)
_add("MLP",           mlp_con_ct.iloc[0]["model_name"],    mlp_con_ct.iloc[0],  ct_scale)

comparison_df = pd.DataFrame(rows)

print("\n" + "="*135)
print("   MODEL COMPARISON TABLE")
print("="*135)
print(comparison_df.to_string(index=False))
print("="*135)
print("\nNote: PPML rows are in LEVELS (raw $) — RMSE/MAE not comparable to log-scale rows. R² is dimensionless and comparable.")
print("For the apples-to-apples comparison vs OLS Linear FE (R²=0.5614 on combined_trade_baci), see the bottom block of ML rows.")

## Step 9: Gravity Residuals vs Conflict — does conflict explain what gravity can't?

Earlier steps showed that adding conflict features adds little predictive power once gravity fundamentals (log GDP, log distance) and country fixed effects are controlled for. That raises the **inference question**: do conflict events still correlate with the *unexplained* part of trade — the residuals from a gravity-only fit?

If conflict matters but gets absorbed by GDP/FE in joint estimation, residuals from a gravity-only model should still correlate with conflict. If the residuals are pure noise, conflict's effect on trade flows entirely through the gravity channel (GDP shocks, institutional capacity captured by FE) — i.e., there is no independent conflict channel.

This cell:
1. Fits a gravity-only XGBoost (`log_trade ~ log_gdp_o + log_gdp_d + log_distw + contig + comlang + iso3 FE`) on train years only
2. Predicts on the full sample → computes residuals
3. Correlates **test-set** residuals with each conflict feature (Pearson)
4. Plots residual vs `fatalities` and `total_conflicts`
5. Lists the top 15 dyad-years where actual trade fell *most below* gravity prediction — narrative case studies for the thesis

**How to read the result:**
- *Strong negative correlation* (residual decreases as conflict rises) → conflict suppresses trade beyond what GDP shocks alone explain. Independent channel exists.
- *Near-zero correlation* → conflict's effect runs through gravity. No independent channel.
- *Top-15 list dominated by known conflict periods* (e.g., Mali 2012+, Burkina Faso 2014+) → narrative evidence for case-study chapters.

In [ ]:
from scipy.stats import pearsonr

# --- 1) Build feature matrix for gravity-only XGBoost (NO conflict features) ---
df_resid = df_log.dropna(subset=["tradeflow_baci"]).copy()
df_resid["log_trade"] = np.log(df_resid["tradeflow_baci"])
df_resid["log_gdp_o"] = np.log(df_resid["gdp_o"])
df_resid["log_gdp_d"] = np.log(df_resid["gdp_d"])
df_resid["log_distw"] = np.log(df_resid["distw_arithmetic"])

fe_dummies = []
for fe_col in ["iso3_o", "iso3_d"]:
    dums = pd.get_dummies(df_resid[fe_col], prefix=fe_col, drop_first=True, dtype=float)
    df_resid = pd.concat([df_resid, dums], axis=1)
    fe_dummies.extend(dums.columns.tolist())

feat_cols = ["log_gdp_o", "log_gdp_d", "log_distw",
             "contig", "comlang_off", "comlang_ethno"] + fe_dummies
df_resid = df_resid.dropna(subset=feat_cols + ["log_trade"]).copy()

# --- 2) Fit on training years only, predict on full sample ---
train_mask = df_resid["year"] <= 2016
gravity_model = XGBRegressor(
    objective="reg:squarederror", n_estimators=500, learning_rate=0.05,
    max_depth=6, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1,
)
gravity_model.fit(df_resid.loc[train_mask, feat_cols],
                  df_resid.loc[train_mask, "log_trade"])

df_resid["pred_log_trade"] = gravity_model.predict(df_resid[feat_cols])
df_resid["residual"]       = df_resid["log_trade"] - df_resid["pred_log_trade"]

print(f"Sample: {len(df_resid)} dyad-years  "
      f"(train: {train_mask.sum()}, test: {(~train_mask).sum()})")
print(f"Residual stats — mean: {df_resid['residual'].mean():.4f}  "
      f"std: {df_resid['residual'].std():.4f}")

# --- 3) Pearson correlations: TEST-set residuals vs each conflict feature ---
# Excludes _roll3 columns (those are derived) — focus on raw conflict signals
conflict_features = [c for c in df_resid.columns
                     if any(c.startswith(p) for p in conflict_prefixes)
                     and not c.endswith("_roll3")] + ["fatalities"]

df_resid["total_conflicts"] = df_resid[conflict_features].sum(axis=1)

test_resid = df_resid[~train_mask].copy()

corr_rows = []
for col in conflict_features + ["total_conflicts"]:
    if test_resid[col].nunique() < 2:
        continue
    r, p = pearsonr(test_resid[col], test_resid["residual"])
    corr_rows.append({"feature": col, "pearson_r": round(r, 4),
                      "p_value": round(p, 4), "abs_r": abs(r)})

corr_df = pd.DataFrame(corr_rows).sort_values("abs_r", ascending=False).drop(columns="abs_r")

print("\n" + "="*80)
print("  Pearson correlation: TEST-set gravity residuals vs conflict features (top 15)")
print("="*80)
print(corr_df.head(15).to_string(index=False))

# --- 4) Scatter plots: residual vs fatalities / total_conflicts ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, xcol, label in [(axes[0], "fatalities",      "log1p(fatalities)"),
                         (axes[1], "total_conflicts", "log1p(total conflicts, summed)")]:
    ax.scatter(test_resid[xcol], test_resid["residual"],
               alpha=0.4, s=15, color="steelblue", edgecolor="white", linewidth=0.5)
    if test_resid[xcol].nunique() >= 2:
        z = np.polyfit(test_resid[xcol], test_resid["residual"], 1)
        xs = np.linspace(test_resid[xcol].min(), test_resid[xcol].max(), 100)
        ax.plot(xs, z[0]*xs + z[1], color="crimson", linewidth=2,
                label=f"slope = {z[0]:.4f}")
        ax.legend()
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
    ax.set_xlabel(label)
    ax.set_ylabel("Residual: log(trade) actual − gravity prediction")
    ax.set_title(f"Residual vs {xcol}  (test set, year > 2016)")

plt.tight_layout()
plt.show()

# --- 5) Top dyad-years where trade was MUCH BELOW gravity prediction ---
display_cols = ["iso3_o", "iso3_d", "year", "log_trade", "pred_log_trade",
                "residual", "fatalities", "total_conflicts"]

top_neg = df_resid.nsmallest(15, "residual")[display_cols]

print("\n" + "="*120)
print("  Top 15 dyad-years where ACTUAL TRADE WAS MUCH BELOW gravity prediction (full sample)")
print("="*120)
print(top_neg.round(3).to_string(index=False))

# --- 6) Top positive residuals (for contrast — where trade exceeded prediction) ---
top_pos = df_resid.nlargest(10, "residual")[display_cols]
print("\n" + "="*120)
print("  Top 10 dyad-years where ACTUAL TRADE WAS MUCH ABOVE gravity prediction")
print("="*120)
print(top_pos.round(3).to_string(index=False))

### Step 9b: Local Anomaly — How Unusual Was the Conflict for *This* Country?

Raw fatalities counts mix two things: countries that are chronically high-conflict (Mali, Burkina Faso) vs years where conflict spiked above that country's own baseline. A Mali-year with `fatalities = 6.8` (log1p) might be near-typical for Mali but extreme for Cape Verde.

This cell adds **per-origin-country z-scores**: for each `iso3_o`, compute the mean and std of `fatalities` and `total_conflicts` across all years, then express each dyad-year as how many standard deviations above/below that country's own history. Then we re-rank residuals against *local anomaly* rather than absolute conflict level.

If conflict has an independent suppressive channel, residuals should correlate more strongly with **local z-scored conflict** than with raw conflict — because the z-score isolates "unexpectedly bad year for this country" from "this country has high baseline conflict" (which the FE already absorbs).

In [ ]:
# --- 1) Per-origin-country baseline (mean / std across all years) ---
country_baseline = df_resid.groupby("iso3_o").agg(
    fatalities_o_mean       = ("fatalities",      "mean"),
    fatalities_o_std        = ("fatalities",      "std"),
    total_conflicts_o_mean  = ("total_conflicts", "mean"),
    total_conflicts_o_std   = ("total_conflicts", "std"),
).reset_index()

# Guard against zero std (single-year countries) — replace with NaN so z-score becomes NaN
country_baseline.loc[country_baseline["fatalities_o_std"]      == 0, "fatalities_o_std"]      = np.nan
country_baseline.loc[country_baseline["total_conflicts_o_std"] == 0, "total_conflicts_o_std"] = np.nan

print("="*100)
print("  Per-origin-country baseline conflict levels (log1p scale)")
print("="*100)
print(country_baseline.round(3).to_string(index=False))

# --- 2) Merge baseline back, compute z-scores ---
df_resid = df_resid.merge(country_baseline, on="iso3_o", how="left")

df_resid["fatalities_local_z"] = (
    (df_resid["fatalities"] - df_resid["fatalities_o_mean"])
    / df_resid["fatalities_o_std"]
)
df_resid["total_conflicts_local_z"] = (
    (df_resid["total_conflicts"] - df_resid["total_conflicts_o_mean"])
    / df_resid["total_conflicts_o_std"]
)

# Refresh the test-set view
test_resid = df_resid[df_resid["year"] > 2016].copy()

# --- 3) Correlation: residuals vs LOCAL Z-SCORE vs RAW conflict ---
def _corr(col):
    sub = test_resid.dropna(subset=[col, "residual"])
    if sub[col].nunique() < 2:
        return np.nan, np.nan
    r, p = pearsonr(sub[col], sub["residual"])
    return r, p

print("\n" + "="*80)
print("  Pearson correlation: TEST-set residuals vs conflict (local z-score vs raw)")
print("="*80)
for col, label in [
    ("fatalities",                 "fatalities (raw log1p)             "),
    ("fatalities_local_z",         "fatalities — LOCAL z-score          "),
    ("total_conflicts",            "total_conflicts (raw log1p)        "),
    ("total_conflicts_local_z",    "total_conflicts — LOCAL z-score     "),
]:
    r, p = _corr(col)
    sig = "**" if (not np.isnan(p) and p < 0.05) else "  "
    print(f"  {sig} {label} :  r = {r:+.4f}   p = {p:.4f}")
print()
print("  ** = p < 0.05.  Compare local-z to raw: if |r_local_z| > |r_raw|, conflict's")
print("  effect is better captured by within-country anomaly than absolute level.")

# --- 4) Top 15 negative residuals — now with local z-scores ---
display_cols = ["iso3_o", "iso3_d", "year", "log_trade", "pred_log_trade", "residual",
                "fatalities", "fatalities_local_z",
                "total_conflicts", "total_conflicts_local_z"]

top_neg = df_resid.nsmallest(15, "residual")[display_cols]

print("\n" + "="*150)
print("  Top 15 dyad-years where TRADE FELL BELOW gravity prediction — with local conflict anomaly")
print("="*150)
print(top_neg.round(3).to_string(index=False))

# --- 5) Top 10 positive residuals (contrast) ---
top_pos = df_resid.nlargest(10, "residual")[display_cols]

print("\n" + "="*150)
print("  Top 10 dyad-years where TRADE EXCEEDED gravity prediction — with local conflict anomaly")
print("="*150)
print(top_pos.round(3).to_string(index=False))

# --- 6) Re-do scatter plots with z-scored conflict on x-axis ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, xcol, label in [
    (axes[0], "fatalities_local_z",      "Fatalities — z-score within origin country"),
    (axes[1], "total_conflicts_local_z", "Total conflicts — z-score within origin country"),
]:
    plot_data = test_resid.dropna(subset=[xcol, "residual"])
    ax.scatter(plot_data[xcol], plot_data["residual"],
               alpha=0.4, s=15, color="steelblue", edgecolor="white", linewidth=0.5)
    if plot_data[xcol].nunique() >= 2:
        z = np.polyfit(plot_data[xcol], plot_data["residual"], 1)
        xs = np.linspace(plot_data[xcol].min(), plot_data[xcol].max(), 100)
        ax.plot(xs, z[0]*xs + z[1], color="crimson", linewidth=2,
                label=f"slope = {z[0]:.4f}")
        ax.legend()
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
    ax.axvline(0, color="black", linewidth=0.5, linestyle="--")
    ax.set_xlabel(label)
    ax.set_ylabel("Residual: log(trade) actual − gravity prediction")
    ax.set_title(f"Residual vs {xcol}  (test set, year > 2016)")

plt.tight_layout()
plt.show()

### Step 9c: Reliable Reporting Subset — Is the Residual Just Measurement Noise?

If the top-15 negative residuals are dominated by *measurement noise* (BACI under-reporting, smuggling shifts), then restricting the sample to dyad-years where trade was *actually observed* — i.e., dropping the synthetic imputations — should:

- **Reduce residual variance** (noise filtered out)
- **Improve R²** on the test set (cleaner targets)
- **Strengthen any real conflict correlation** (signal-to-noise ratio improves)

If instead R² is similar and correlations don't move, the noise hypothesis is wrong and the residuals reflect something the gravity model genuinely can't capture.

In [ ]:
# --- 1) Filter to non-synthetic BACI rows only ---
reliable_mask = (df_log.get("tradeflow_baci_synthetic_flag", 0) == 0) & df_log["tradeflow_baci"].notna()
df_reliable = df_log[reliable_mask].copy()

print(f"Full sample:       {len(df_log):>6,} rows")
print(f"Reliable subset:   {len(df_reliable):>6,} rows  ({len(df_reliable)/len(df_log)*100:.1f}%)")

# --- 2) Re-build features on the reliable subset ---
df_reliable["log_trade"] = np.log(df_reliable["tradeflow_baci"])
df_reliable["log_gdp_o"] = np.log(df_reliable["gdp_o"])
df_reliable["log_gdp_d"] = np.log(df_reliable["gdp_d"])
df_reliable["log_distw"] = np.log(df_reliable["distw_arithmetic"])

fe_dummies_r = []
for fe_col in ["iso3_o", "iso3_d"]:
    dums = pd.get_dummies(df_reliable[fe_col], prefix=fe_col, drop_first=True, dtype=float)
    df_reliable = pd.concat([df_reliable, dums], axis=1)
    fe_dummies_r.extend(dums.columns.tolist())

feat_cols_r = ["log_gdp_o", "log_gdp_d", "log_distw",
               "contig", "comlang_off", "comlang_ethno"] + fe_dummies_r
df_reliable = df_reliable.dropna(subset=feat_cols_r + ["log_trade"]).copy()

# --- 3) Re-fit gravity-only XGBoost ---
train_mask_r = df_reliable["year"] <= 2016
gravity_model_r = XGBRegressor(
    objective="reg:squarederror", n_estimators=500, learning_rate=0.05,
    max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
)
gravity_model_r.fit(df_reliable.loc[train_mask_r, feat_cols_r],
                    df_reliable.loc[train_mask_r, "log_trade"])

df_reliable["pred_log_trade"] = gravity_model_r.predict(df_reliable[feat_cols_r])
df_reliable["residual"]       = df_reliable["log_trade"] - df_reliable["pred_log_trade"]

# Total conflicts on reliable subset
conflict_features_r = [c for c in df_reliable.columns
                       if any(c.startswith(p) for p in conflict_prefixes)
                       and not c.endswith("_roll3")] + ["fatalities"]
df_reliable["total_conflicts"] = df_reliable[conflict_features_r].sum(axis=1)

test_resid_r = df_reliable[~train_mask_r].copy()
test_resid_full = test_resid  # from Step 9b

# --- 4) Side-by-side comparison ---
def _stats(df_subset):
    return {
        "n_test":  len(df_subset),
        "r2":      r2_score(df_subset["log_trade"], df_subset["pred_log_trade"]),
        "rmse":    root_mean_squared_error(df_subset["log_trade"], df_subset["pred_log_trade"]),
        "resid_std": df_subset["residual"].std(),
    }

s_full     = _stats(test_resid_full)
s_reliable = _stats(test_resid_r)

print("\n" + "="*85)
print("  GRAVITY MODEL: Full sample vs Reliable-only subset (test set, year > 2016)")
print("="*85)
print(f"  {'Metric':<25}{'Full sample':>20}{'Reliable only':>20}{'Δ':>15}")
print(f"  {'-'*25}{'-'*20}{'-'*20}{'-'*15}")
print(f"  {'Test sample size':<25}{s_full['n_test']:>20,}{s_reliable['n_test']:>20,}{s_reliable['n_test']-s_full['n_test']:>+15,}")
print(f"  {'Test R²':<25}{s_full['r2']:>20.4f}{s_reliable['r2']:>20.4f}{s_reliable['r2']-s_full['r2']:>+15.4f}")
print(f"  {'Test RMSE':<25}{s_full['rmse']:>20.4f}{s_reliable['rmse']:>20.4f}{s_reliable['rmse']-s_full['rmse']:>+15.4f}")
print(f"  {'Residual std':<25}{s_full['resid_std']:>20.4f}{s_reliable['resid_std']:>20.4f}{s_reliable['resid_std']-s_full['resid_std']:>+15.4f}")

# --- 5) Conflict correlations on reliable subset ---
print("\n" + "="*80)
print("  Pearson correlation: residuals vs conflict on RELIABLE subset (test only)")
print("="*80)
for col in ["fatalities", "total_conflicts"]:
    sub = test_resid_r.dropna(subset=[col, "residual"])
    if sub[col].nunique() < 2:
        continue
    r, p = pearsonr(sub[col], sub["residual"])
    sig = "**" if p < 0.05 else "  "
    # also report full-sample for comparison
    sub_full = test_resid_full.dropna(subset=[col, "residual"])
    r_full, p_full = pearsonr(sub_full[col], sub_full["residual"])
    print(f"  {sig} {col:<25}: r_reliable = {r:+.4f}  (p={p:.4f})   |   r_full = {r_full:+.4f}  (p={p_full:.4f})")

print("\n  If R² rises and residual std falls on reliable subset → the residuals were largely measurement noise.")
print("  If correlations stay near zero → conflict has no detectable independent channel even on clean data.")

### Step 9d: Individual Conflict Types — Where Is the Signal Hiding?

`total_conflicts` is an aggregate that mixes things with very different economic implications:

- **Direct violence** (`event_Battles`, `event_Violence against civilians`, `target_Civilians`) — disrupts roads, ports, supply chains
- **Political activity** (`event_Protests`, `disorder_Demonstrations`, `event_Strategic developments`) — concentrated in capitals, mostly non-disruptive to trade
- **Identity violence** (`perpetrator_Identity militia`, `perpetrator_Rebel group`) — geographically concentrated but high-impact

Aggregating them washes out signal. This cell tests each subcategory independently, ranks them by signed correlation with residuals, and highlights the violence-vs-protest contrast. **If the suppression hypothesis is right, `event_Battles` and civilian-violence indicators should show negative correlations while protests should be near zero or positive.**

In [ ]:
# --- 1) All raw conflict columns (no rolling derivatives) ---
all_conflict_cols = [c for c in df_resid.columns
                     if any(c.startswith(p) for p in ["disorder_", "event_", "perpetrator_", "target_"])
                     and not c.endswith("_roll3")] + ["fatalities"]

# --- 2) Compute correlation of each with TEST-set residuals ---
corr_rows = []
for col in all_conflict_cols:
    sub = test_resid.dropna(subset=[col, "residual"])
    if sub[col].nunique() < 2:
        continue
    r, p = pearsonr(sub[col], sub["residual"])
    corr_rows.append({
        "feature":     col,
        "category":    col.split("_")[0] if "_" in col else "other",
        "pearson_r":   r,
        "p_value":     p,
        "abs_r":       abs(r),
        "significant": p < 0.05,
    })

corr_full = pd.DataFrame(corr_rows).sort_values("pearson_r").reset_index(drop=True)

print("="*100)
print("  Pearson correlation: TEST-set residuals vs each conflict subcategory")
print("="*100)
print("  Sorted by signed r — most negative first (strongest suppression signal)")
print()
disp = corr_full[["feature", "pearson_r", "p_value", "significant"]].copy()
disp["pearson_r"] = disp["pearson_r"].round(4)
disp["p_value"]   = disp["p_value"].round(4)
print(disp.to_string(index=False))

# --- 3) Bar chart of correlations, color-coded by significance ---
fig, ax = plt.subplots(figsize=(10, max(7, len(corr_full) * 0.32)))
colors = ["crimson" if s else "lightsteelblue" for s in corr_full["significant"]]
ax.barh(corr_full["feature"], corr_full["pearson_r"],
        color=colors, alpha=0.9, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson r — TEST-set gravity residual vs feature")
ax.set_title("Residual correlation by conflict subcategory  (red = p < 0.05)")
plt.tight_layout()
plt.show()

# --- 4) Violence vs Protest comparison ---
print("\n" + "="*90)
print("  KEY COMPARISON: violence indicators vs political-activity indicators")
print("="*90)

key_groups = {
    "VIOLENCE / DISRUPTION": [
        ("event_Battles",                       "Direct combat"),
        ("event_Violence against civilians",    "Civilian violence"),
        ("event_Explosions/Remote violence",    "Explosions / remote attacks"),
        ("target_Civilians",                    "Civilians targeted"),
        ("perpetrator_Rebel group",             "Rebel group activity"),
        ("perpetrator_Identity militia",        "Identity militia activity"),
        ("fatalities",                          "Fatalities"),
    ],
    "POLITICAL ACTIVITY (capital-bound)": [
        ("event_Protests",                      "Protests"),
        ("event_Riots",                         "Riots"),
        ("event_Strategic developments",        "Strategic developments"),
        ("disorder_Demonstrations",             "Demonstrations"),
        ("perpetrator_Protesters",              "Protesters as actor"),
        ("perpetrator_Rioters",                 "Rioters as actor"),
    ],
}

for group_name, feats in key_groups.items():
    print(f"\n  --- {group_name} ---")
    for feat, label in feats:
        if feat in corr_full["feature"].values:
            row = corr_full[corr_full["feature"] == feat].iloc[0]
            sig = "**" if row["significant"] else "  "
            print(f"    {sig} {label:<35}({feat:<40}): r = {row['pearson_r']:+.4f}  p = {row['p_value']:.4f}")

print("\n  Read: if violence indicators are more negative than political ones, the suppression channel")
print("  exists — it just gets averaged out in the aggregate. If both groups are near zero, no channel.")

### Step 9e: Case Study — Trade Trajectories vs Gravity Predictions

Aggregate correlations summarize the average. Case studies reveal *where* the gravity model breaks down. For each origin country, we plot **actual log-trade vs gravity prediction** to its top trading partners, with key political events marked.

The visual signature of a conflict shock would be: actual trade tracks the gravity prediction, then diverges sharply at the event year, then either recovers (J-curve) or stays depressed. Visual evidence is more compelling for a thesis than a correlation coefficient — even if the correlation is small in aggregate, individual country histories may show clear shocks.

Three case studies:
- **Mali** — 2012 separatist crisis + coup, ongoing jihadist insurgency, 2020/21 coups
- **Burkina Faso** — 2014 Compaoré ouster, 2022 coup, expanding insecurity
- **The Gambia** — 2016 constitutional crisis, peaceful 2017 transition

In [ ]:
def plot_country_trade_trajectory(df_resid, origin, events=None, max_partners=8):
    """
    Plot log(trade) vs gravity prediction for `origin` to its top destinations over time.
    Shaded area = residual (actual - predicted). Vertical orange lines mark event years.
    """
    sub = df_resid[df_resid["iso3_o"] == origin].copy()
    if sub.empty:
        print(f"  No data for {origin}")
        return

    partner_volume = sub.groupby("iso3_d")["log_trade"].sum().sort_values(ascending=False)
    top_partners = partner_volume.head(max_partners).index.tolist()

    n = len(top_partners)
    ncols = 4
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), sharex=True)
    axes = np.atleast_2d(axes).flatten()

    for ax, dest in zip(axes, top_partners):
        d = sub[sub["iso3_d"] == dest].sort_values("year")
        if d.empty:
            ax.axis("off")
            continue

        ax.plot(d["year"], d["log_trade"],     "o-", color="steelblue", markersize=4, label="actual")
        ax.plot(d["year"], d["pred_log_trade"], "x--", color="crimson",  markersize=5, label="gravity pred")
        ax.fill_between(d["year"], d["log_trade"], d["pred_log_trade"],
                        alpha=0.15, color="gray")

        if events:
            ymin, ymax = ax.get_ylim()
            for yr, label in events:
                if d["year"].min() <= yr <= d["year"].max():
                    ax.axvline(yr, color="orange", linewidth=1.4,
                               linestyle=":", alpha=0.8)
                    ax.text(yr, ymax, f" {label}", fontsize=7,
                            rotation=90, ha="left", va="top",
                            color="darkorange")

        # Train/test split visual cue
        ax.axvline(2016.5, color="black", linewidth=0.6, alpha=0.4)

        ax.set_title(f"{origin} → {dest}", fontsize=10)
        ax.set_xlabel("Year")
        ax.set_ylabel("log(trade)")
        ax.legend(fontsize=7, loc="lower right")
        ax.grid(alpha=0.3)

    for ax in axes[n:]:
        ax.axis("off")

    plt.suptitle(f"{origin} as exporter: actual vs gravity-predicted trade  "
                 f"(orange = political events, black line = train/test split)",
                 fontsize=12, y=1.00)
    plt.tight_layout()
    plt.show()


# === MALI ===
print("="*90)
print("  CASE STUDY 1: MALI — 2012 separatist crisis & coup, jihadist insurgency, 2020/21 coups")
print("="*90)
plot_country_trade_trajectory(
    df_resid, "MLI",
    events=[
        (2012, "Tuareg+coup"),
        (2013, "Serval intervention"),
        (2015, "Algiers Accord"),
        (2020, "Aug 2020 coup"),
        (2021, "May 2021 coup"),
    ],
)

# === BURKINA FASO ===
print("\n" + "="*90)
print("  CASE STUDY 2: BURKINA FASO — 2014 Compaoré ouster, 2015 coup attempt, 2022 coup")
print("="*90)
plot_country_trade_trajectory(
    df_resid, "BFA",
    events=[
        (2014, "Compaoré ousted"),
        (2015, "Sept coup attempt"),
        (2022, "Jan 2022 coup"),
    ],
)

# === THE GAMBIA ===
print("\n" + "="*90)
print("  CASE STUDY 3: THE GAMBIA — 2016 constitutional crisis, 2017 peaceful transition")
print("="*90)
plot_country_trade_trajectory(
    df_resid, "GMB",
    events=[
        (2016, "Constitutional crisis"),
        (2017, "Jammeh exits"),
    ],
)

# === Summary: residual at event-year for each case study ===
print("\n" + "="*90)
print("  RESIDUAL AT EVENT YEARS — average residual across all destinations")
print("="*90)
event_summary = []
for origin, evts in [
    ("MLI", [2012, 2013, 2015, 2020, 2021]),
    ("BFA", [2014, 2015, 2022]),
    ("GMB", [2016, 2017]),
]:
    for yr in evts:
        sub = df_resid[(df_resid["iso3_o"] == origin) & (df_resid["year"] == yr)]
        if len(sub) == 0:
            continue
        event_summary.append({
            "country":   origin,
            "year":      yr,
            "n_dyads":   len(sub),
            "mean_residual": round(sub["residual"].mean(), 4),
            "median_residual": round(sub["residual"].median(), 4),
            "fatalities_log1p": round(sub["fatalities"].mean(), 3),
        })

print(pd.DataFrame(event_summary).to_string(index=False))
print("\n  Negative mean_residual at an event year = trade fell below gravity expectation that year.")
print("  Compare to that country's overall residual std (~1.5) to gauge magnitude.")

## Step 10: All Four Models on the Reliable Subset of `combined_trade_baci`

The reliable-subset filter (`Step 9c`) showed that gravity-only XGBoost on `tradeflow_baci` jumps from R² = 0.42 → 0.61 once synthetic-imputed rows are dropped. This step extends that filter to `combined_trade_baci` — the smoothest, least-noisy target — and runs **all four model families** (XGBoost, Random Forest, MLP, OLS Linear FE) on the resulting clean subset.

Filter: `(all source synthetic_flag == 0)` — i.e., rows where BACI, IMF (origin and destination), and Comtrade (origin and destination) all reported the trade flow directly. This is the strictest "fully observed" subset.

For each model, two specifications: gravity-only and the best-performing conflict ablation. The OLS Linear FE result on this same subset is the natural baseline to compare ML against — apples-to-apples on target, sample, and feature set.

**The expected payoff:** if the ML-vs-OLS-FE gap closes on this clean subset, gravity is the dominant explanation and ML adds little. If ML opens a meaningful lead, nonlinear interactions (or dyad-specific patterns the FE doesn't catch) are doing genuine work.

In [ ]:
from sklearn.linear_model import LinearRegression

# --- OLS Linear FE function (mirrors structure of xgboost_regression for combo-wrapper compatibility) ---
def baseline_linear_fe_inline(df, target_variable, columns, year_split=2016):
    """
    OLS gravity baseline with iso3_o + iso3_d fixed effects (sklearn LinearRegression).
    Same preprocessing pipeline as the ML functions for apples-to-apples comparison.
    """
    columns = list(dict.fromkeys(columns))

    missing = required_columns - set(columns)
    if missing:
        raise ValueError(f"Function columns MUST include all of: {required_columns}")
    if target_variable not in legal_target_vars:
        raise ValueError(f"Invalid target variable '{target_variable}'")

    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].dropna().copy()
    test_df  = test_df_full[all_needed].dropna().copy()

    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    for raw, logged in [("gdp_o", "log_gdp_o"), ("gdp_d", "log_gdp_d"), ("distw_arithmetic", "log_distw")]:
        train_df[logged] = np.log(train_df[raw])
        test_df[logged]  = np.log(test_df[raw])

    update_list = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list), 2):
        columns.remove(update_list[i])
        columns.append(update_list[i+1])

    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float)
        dummies_test  = pd.get_dummies(test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float)
        dummies_test  = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)
        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]
    X_test  = test_df_model[columns]
    y_test  = test_df_model[f"{target_variable}_log_trade"]

    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"OLS-FE Out-of-sample RMSE: {rmse:.4f}")
    print(f"OLS-FE Out-of-sample MAE:  {mae:.4f}")
    print(f"OLS-FE Out-of-sample R²:   {r2:.4f}")

    return {"model_name": f"OLS_FE_{target_variable}", "n_train": len(X_train), "n_test": len(X_test),
            "rmse": rmse, "mae": mae, "r2": r2}


# --- OLS FE conflict ablation wrapper (mirrors find_best_conflict_combo1/2/3) ---
def find_best_conflict_combo4(df, target_variable, base_columns=None, year_split=2016):
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    cprefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in cprefixes)] + ["fatalities"]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }
    base_groups = list(conflict_groups.keys())
    combos = []
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos.append((" + ".join(combo), extra))
    all_cols = [c for g in base_groups for c in conflict_groups[g]] + ["fatalities"]
    combos.append(("disorder + event + perpetrator + target + fatalities", all_cols))
    combos.append(("fatalities", ["fatalities"]))
    combos.append(("total_conflicts", ["total_conflicts"]))
    combos.append(("total_conflicts + fatalities", ["total_conflicts", "fatalities"]))

    results = []
    for label, extra_cols in combos:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = baseline_linear_fe_inline(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)
    display_df = results_df.copy()
    for c in ("rmse", "mae", "r2"):
        display_df[c] = display_df[c].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — OLS FE on {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")
    return results_df


# --- Build the reliable subset of combined_trade_baci ---
syn_flag_cols = [c for c in df_log.columns if c.endswith("_synthetic_flag")]
mask_all_real = (df_log[syn_flag_cols] == 0).all(axis=1) & df_log["combined_trade_baci"].notna()
df_reliable_combined = df_log[mask_all_real].copy()

print(f"df_log full sample:                {len(df_log):>6,} rows")
print(f"Fully-observed combined subset:    {len(df_reliable_combined):>6,} rows  ({len(df_reliable_combined)/len(df_log)*100:.1f}%)")
print(f"Train (year ≤ 2016): {(df_reliable_combined['year'] <= 2016).sum()}")
print(f"Test  (year > 2016): {(df_reliable_combined['year'] > 2016).sum()}")

In [ ]:
base_cols = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

# === OLS Linear FE ===
print("="*100)
print("  OLS LINEAR FE — gravity only")
print("="*100)
ols_nocon_rc = baseline_linear_fe_inline(df_reliable_combined, "combined_trade_baci", columns=base_cols)

print("\n" + "="*100)
print("  OLS LINEAR FE — full conflict ablation")
print("="*100)
ols_con_rc = find_best_conflict_combo4(df_reliable_combined, "combined_trade_baci")

# === XGBoost ===
print("="*100)
print("  XGBOOST — gravity only")
print("="*100)
XG_nocon_rc = xgboost_regression(df_reliable_combined, "combined_trade_baci", columns=base_cols)

print("\n" + "="*100)
print("  XGBOOST — full conflict ablation")
print("="*100)
xgb_con_rc = find_best_conflict_combo1(df_reliable_combined, "combined_trade_baci")

# === Random Forest ===
print("="*100)
print("  RANDOM FOREST — gravity only")
print("="*100)
RFR_nocon_rc = random_forest_regression(df_reliable_combined, "combined_trade_baci", columns=base_cols)

print("\n" + "="*100)
print("  RANDOM FOREST — full conflict ablation")
print("="*100)
RFR_con_rc = find_best_conflict_combo2(df_reliable_combined, "combined_trade_baci")

# === MLP ===
print("="*100)
print("  MLP — gravity only")
print("="*100)
mlp_nocon_rc = multilayer_perceptron(df_reliable_combined, "combined_trade_baci", columns=base_cols)

print("\n" + "="*100)
print("  MLP — full conflict ablation")
print("="*100)
mlp_con_rc = find_best_conflict_combo3(df_reliable_combined, "combined_trade_baci")

In [ ]:
# === Step 10 final comparison table ===
rows_s10 = []

def _add_s10(model, conflict, res):
    rows_s10.append({
        "Model": model,
        "Conflict": conflict,
        "N Train": res["n_train"],
        "N Test":  res["n_test"],
        "RMSE":    round(res["rmse"], 4),
        "MAE":     round(res["mae"],  4),
        "R²":      round(res["r2"],   4),
    })

_add_s10("OLS Linear FE",        "none",                              ols_nocon_rc)
_add_s10("OLS Linear FE",        ols_con_rc.iloc[0]["model_name"],    ols_con_rc.iloc[0])
_add_s10("XGBoost",              "none",                              XG_nocon_rc)
_add_s10("XGBoost",              xgb_con_rc.iloc[0]["model_name"],    xgb_con_rc.iloc[0])
_add_s10("Random Forest",        "none",                              RFR_nocon_rc)
_add_s10("Random Forest",        RFR_con_rc.iloc[0]["model_name"],    RFR_con_rc.iloc[0])
_add_s10("MLP",                  "none",                              mlp_nocon_rc)
_add_s10("MLP",                  mlp_con_rc.iloc[0]["model_name"],    mlp_con_rc.iloc[0])

step10_df = pd.DataFrame(rows_s10)

print("\n" + "="*120)
print(f"   STEP 10 COMPARISON — All models on RELIABLE combined_trade_baci  (N test = {ols_nocon_rc['n_test']})")
print("="*120)
print(step10_df.to_string(index=False))
print("="*120)
print()
print("  Read: gap between OLS Linear FE and best ML R² indicates whether nonlinearity matters")
print("  beyond the log-linear gravity functional form on this clean target.")

## Step 11: Bootstrap Confidence Interval on the RF-vs-OLS-FE R² Gap

The Step 10 result — Random Forest R² = 0.6749 vs OLS Linear FE R² = 0.6140 — gives a +0.061 gap on **N test = 275**. This is small enough that a reviewer might (correctly) ask whether the gap is statistically distinguishable from zero.

This cell answers that question by bootstrap. We refit RF and OLS Linear FE on the same training data (year ≤ 2016 of the reliable subset), generate predictions on the test set, then resample the test set with replacement 2,000 times. For each bootstrap replicate we compute (RF R² − OLS R²). The resulting empirical distribution gives a 95% CI on the gap.

**Read the result:** if the 95% CI excludes zero, the RF lift is robust; if not, the gap could plausibly be a sampling artifact and Statement 1b in `findings_summary.md` should be softened.

In [ ]:
# --- Shared preprocessing (mirrors xgboost_regression / baseline_linear_fe_inline) ---
def _prep_log_fe(df, target, base):
    cols = list(dict.fromkeys(base))
    train = df[df["year"] <= 2016].copy()
    test  = df[df["year"] >  2016].copy()
    needed = cols + [target]
    train = train[needed + ["iso3_o", "iso3_d"]].dropna(subset=needed).copy()
    test  = test[needed + ["iso3_o", "iso3_d"]].dropna(subset=needed).copy()
    train = train[train[target] > 0]
    test  = test[test[target] > 0]
    train["log_y"] = np.log(train[target])
    test["log_y"]  = np.log(test[target])
    for raw, logged in [("gdp_o","log_gdp_o"), ("gdp_d","log_gdp_d"), ("distw_arithmetic","log_distw")]:
        train[logged] = np.log(train[raw])
        test[logged]  = np.log(test[raw])
    feat = ["log_gdp_o", "log_gdp_d", "log_distw"] + [c for c in cols if c not in ("gdp_o","gdp_d","distw_arithmetic")]
    for fe in ["iso3_o", "iso3_d"]:
        d_tr = pd.get_dummies(train[fe], prefix=fe, drop_first=True, dtype=float)
        d_te = pd.get_dummies(test[fe],  prefix=fe, drop_first=True, dtype=float).reindex(columns=d_tr.columns, fill_value=0)
        train = pd.concat([train, d_tr], axis=1)
        test  = pd.concat([test,  d_te], axis=1)
        feat.extend(d_tr.columns.tolist())
    return train[feat], train["log_y"], test[feat], test["log_y"]


# --- Refit RF and OLS Linear FE on the same data Step 10 used ---
# IMPORTANT: match random_forest_regression's hyperparameters exactly (max_features="sqrt")
X_tr, y_tr, X_te, y_te = _prep_log_fe(df_reliable_combined, "combined_trade_baci", base_cols)

rf_boot = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features="sqrt",        # ← was missing in the previous version (caused RF to collapse)
    random_state=42,
    n_jobs=-1,
)
rf_boot.fit(X_tr, y_tr)
rf_pred = rf_boot.predict(X_te)

ols_boot = LinearRegression()
ols_boot.fit(X_tr, y_tr)
ols_pred = ols_boot.predict(X_te)

r2_rf_full  = r2_score(y_te, rf_pred)
r2_ols_full = r2_score(y_te, ols_pred)

print(f"Refitted on N train = {len(X_tr)}, N test = {len(X_te)}")
print(f"  RF  R² (full test set): {r2_rf_full:.4f}    (Step 10 reported 0.6749 — should match)")
print(f"  OLS R² (full test set): {r2_ols_full:.4f}    (Step 10 reported 0.6140 — should match)")
print(f"  Point gap:              {r2_rf_full - r2_ols_full:+.4f}")

# --- Bootstrap the gap ---
n_boot   = 2000
n_test   = len(y_te)
y_te_arr = y_te.values
rf_arr   = np.asarray(rf_pred)
ols_arr  = np.asarray(ols_pred)

rng = np.random.default_rng(42)
deltas = np.empty(n_boot)
for i in range(n_boot):
    idx = rng.integers(0, n_test, size=n_test)
    deltas[i] = r2_score(y_te_arr[idx], rf_arr[idx]) - r2_score(y_te_arr[idx], ols_arr[idx])

ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])
mean_delta      = deltas.mean()
pr_pos          = (deltas > 0).mean()

# Correct verdict logic
if ci_low > 0:
    verdict = "CI EXCLUDES zero (entirely positive — RF beats OLS)"
elif ci_high < 0:
    verdict = "CI EXCLUDES zero (entirely negative — OLS beats RF)"
else:
    verdict = "CI INCLUDES zero (gap not statistically distinguishable from 0)"

print(f"\n=== BOOTSTRAP RESULT ({n_boot:,} resamples) ===")
print(f"  Mean Δ (RF − OLS) R²:     {mean_delta:+.4f}")
print(f"  95% CI:                   [{ci_low:+.4f}, {ci_high:+.4f}]")
print(f"  Pr(Δ > 0):                {pr_pos:.4f}")
print(f"  Verdict:                  {verdict}")

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(deltas, bins=40, color="steelblue", alpha=0.85, edgecolor="white")
ax.axvline(0, color="black", linewidth=1.5, label="No difference")
ax.axvline(mean_delta, color="crimson", linewidth=2, label=f"Mean Δ = {mean_delta:+.4f}")
ax.axvline(ci_low,  color="darkorange", linewidth=1.2, linestyle="--",
           label=f"95% CI = [{ci_low:+.4f}, {ci_high:+.4f}]")
ax.axvline(ci_high, color="darkorange", linewidth=1.2, linestyle="--")
ax.set_xlabel("R² gap (Random Forest − OLS Linear FE)")
ax.set_ylabel("Bootstrap frequency")
ax.set_title(f"Bootstrap of RF-vs-OLS-FE R² gap ({n_boot:,} resamples, N test = {n_test})\nReliable combined_trade_baci")
ax.legend()
plt.tight_layout()
plt.show()

## Step 12: PPML on Reliable `combined_trade_baci` — Five-Estimator Comparison

The Step 10 comparison table includes OLS Linear FE, XGBoost, Random Forest, and MLP — but not **PPML**, the canonical gravity-literature estimator (Santos Silva & Tenreyro 2006; Yotov 2022). Adding PPML to the same reliable subset closes the methodological loop: it lets us state directly whether nonlinear ML beats not just OLS-FE but also PPML-FE, the strongest gravity-literature benchmark.

PPML predicts in **levels** (raw $) by design. To compare with the log-scale ML results, we report metrics on both scales:

- **Levels**: native PPML output, comparable to the PPML results in `04_modelling copy.ipynb`
- **Log scale**: take log of PPML predictions, then compute R² against log of actual trade — this is the apples-to-apples comparison with OLS-FE / XGBoost / RF / MLP

If RF still beats PPML on the log-scale metric, the methodological argument from Step 10 is strengthened: **the strongest gravity-literature estimator (PPML+FE, advocated by every gravity textbook in your `coll/`) loses to a Random Forest on clean ECOWAS data.**

In [ ]:
def baseline_ppml_fe_inline(df, target_variable, columns, year_split=2016):
    """PPML gravity with iso3_o + iso3_d FE. Returns BOTH levels and log-scale metrics."""
    columns = list(dict.fromkeys(columns))

    missing = required_columns - set(columns)
    if missing:
        raise ValueError(f"Function columns MUST include all of: {required_columns}")
    if target_variable not in legal_target_vars:
        raise ValueError(f"Invalid target variable '{target_variable}'")

    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] >  year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].dropna().copy()
    test_df  = test_df_full[all_needed].dropna().copy()

    # Keep positive trade only (PPML handles zeros, but log-scale comparison cannot)
    train_df = train_df[train_df[target_variable] > 0].copy()
    test_df  = test_df[test_df[target_variable] > 0].copy()

    for raw, logged in [("gdp_o","log_gdp_o"), ("gdp_d","log_gdp_d"), ("distw_arithmetic","log_distw")]:
        train_df[logged] = np.log(train_df[raw])
        test_df[logged]  = np.log(test_df[raw])

    update = ["gdp_o","log_gdp_o","gdp_d","log_gdp_d","distw_arithmetic","log_distw"]
    for i in range(0, len(update), 2):
        columns.remove(update[i])
        columns.append(update[i+1])

    for fe_col in ["iso3_o", "iso3_d"]:
        d_tr = pd.get_dummies(train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float)
        d_te = pd.get_dummies(test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float)
        d_te = d_te.reindex(columns=d_tr.columns, fill_value=0)
        train_df = pd.concat([train_df, d_tr], axis=1)
        test_df  = pd.concat([test_df,  d_te], axis=1)
        columns.extend(d_tr.columns.tolist())

    X_train = sm.add_constant(train_df[columns])
    y_train = train_df[target_variable]
    X_test  = sm.add_constant(test_df[columns]).reindex(columns=X_train.columns, fill_value=0)
    y_test  = test_df[target_variable]

    ppml = sm.GLM(y_train, X_train, family=sm.families.Poisson(sm.families.links.Log()))
    res  = ppml.fit(cov_type="cluster",
                    cov_kwds={"groups": train_df_full.loc[train_df.index, "dyad"]})

    y_pred_levels = np.asarray(res.predict(X_test))

    rmse_l = root_mean_squared_error(y_test, y_pred_levels)
    mae_l  = mean_absolute_error(y_test, y_pred_levels)
    r2_l   = r2_score(y_test, y_pred_levels)

    log_y_test = np.log(y_test.values)
    log_y_pred = np.log(np.maximum(y_pred_levels, 1e-12))
    rmse_log = root_mean_squared_error(log_y_test, log_y_pred)
    mae_log  = mean_absolute_error(log_y_test, log_y_pred)
    r2_log   = r2_score(log_y_test, log_y_pred)

    print(f"PPML FE on {target_variable}")
    print(f"  Levels    : RMSE={rmse_l:>14,.4f}   MAE={mae_l:>14,.4f}   R²={r2_l:.4f}")
    print(f"  Log scale : RMSE={rmse_log:>14.4f}   MAE={mae_log:>14.4f}   R²={r2_log:.4f}   ← apples-to-apples")

    return {
        "model_name":  f"PPML_FE_{target_variable}",
        "n_train":     len(X_train),
        "n_test":      len(X_test),
        "rmse":        rmse_log,   # log-scale for comparison
        "mae":         mae_log,
        "r2":          r2_log,
        "rmse_levels": rmse_l,
        "mae_levels":  mae_l,
        "r2_levels":   r2_l,
    }

# --- Run PPML on the reliable subset of combined_trade_baci ---
print("="*100)
print("  PPML FE — gravity only on RELIABLE combined_trade_baci")
print("="*100)
ppml_nocon_rc = baseline_ppml_fe_inline(df_reliable_combined, "combined_trade_baci", columns=base_cols)

# --- Updated five-estimator comparison ---
ranking = [
    ("Random Forest",  "none",                             RFR_nocon_rc),
    ("Random Forest",  RFR_con_rc.iloc[0]["model_name"],   RFR_con_rc.iloc[0]),
    ("OLS Linear FE",  "disorder (best)",                  ols_con_rc.iloc[0]),
    ("OLS Linear FE",  "none",                             ols_nocon_rc),
    ("MLP",            "fatalities (best)",                mlp_con_rc.iloc[0]),
    ("MLP",            "none",                             mlp_nocon_rc),
    ("XGBoost",        "none",                             XG_nocon_rc),
    ("PPML FE",        "none (log scale)",                 ppml_nocon_rc),
]

print("\n" + "="*120)
print(f"   STEP 12 — Five-estimator comparison on RELIABLE combined_trade_baci  (sorted by log-scale R², N test = {ppml_nocon_rc['n_test']})")
print("="*120)
print(f"  {'Rank':<5}{'Model':<22}{'Conflict':<25}{'RMSE':>12}{'MAE':>12}{'R² (log)':>12}")
print("-"*120)

ranking_sorted = sorted(ranking, key=lambda x: -x[2]["r2"])
for rank_i, (model, conf, res) in enumerate(ranking_sorted, 1):
    print(f"  {rank_i:<5}{model:<22}{conf:<25}{res['rmse']:>12.4f}{res['mae']:>12.4f}{res['r2']:>12.4f}")
print("="*120)
print(f"\n  PPML levels metrics (for comparison with 04_modelling copy.ipynb):")
print(f"    RMSE = {ppml_nocon_rc['rmse_levels']:,.2f}   MAE = {ppml_nocon_rc['mae_levels']:,.2f}   R² = {ppml_nocon_rc['r2_levels']:.4f}")

## Step 13: Cross-CSV Robustness Grid — All Models × All Targets × All CSVs

Twelve dataset variants (3 lags × {non-synthetic, synth threshold 0, synth threshold 10, synth threshold 100} = 3 + 9 = 12 CSVs) crossed with five estimators and six trade targets. For each cell we run the full conflict ablation (~19 combos), giving roughly **6,840 model fits** total.

**Grid:**
- **CSVs (12)**: `ecowas_df_full_lag_{zero,one,two}.csv` (3 non-synthetic) + `ecowas_df_synthetic_full_lag_{zero,one,two}_{0,10,100}.csv` (9 synthetic)
- **Models (5)**: XGBoost, Random Forest, MLP, OLS Linear FE, PPML FE
- **Targets (6)**: `tradeflow_baci`, `tradeflow_imf_o`, `tradeflow_imf_d`, `tradeflow_comtrade_o`, `tradeflow_comtrade_d`, `combined_trade_baci`
- **Base columns (logged inside each model)**: `gdp_o`, `gdp_d`, `distw_arithmetic`, `contig`, `comlang_off`, `comlang_ethno`

**Behaviour:**
- Each conflict column is `log1p`-transformed on load (matching the rest of the notebook's preprocessing convention)
- Per-model stdout is silenced — there would be ~360 ranking tables otherwise
- Results are written incrementally to `results/robustness_grid.csv` after each (CSV × model × target) cell, so the run can be **stopped and resumed** without losing progress
- Any cell that errors (e.g., a target column not present in a particular CSV) is skipped and logged

**Estimated runtime: 3–8 hours.** PPML and Random Forest are the slowest. MLP convergence varies. Run during a coffee break or overnight.

In [ ]:
import time
from pathlib import Path

# --- 1) PPML conflict-ablation wrapper (mirrors find_best_conflict_combo1/2/3/4) ---
def find_best_conflict_combo5(df, target_variable, base_columns=None, year_split=2016):
    """PPML FE conflict ablation — same combo logic as combo1-4."""
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    cprefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in cprefixes)]
    if "fatalities" in df.columns:
        all_conflict_cols.append("fatalities")
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1) if all_conflict_cols else 0

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }
    base_groups = list(conflict_groups.keys())
    combos = []
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos.append((" + ".join(combo), extra))
    all_cols = [c for g in base_groups for c in conflict_groups[g]]
    if "fatalities" in df.columns:
        all_cols.append("fatalities")
        combos.append(("disorder + event + perpetrator + target + fatalities", all_cols))
        combos.append(("fatalities", ["fatalities"]))
    combos.append(("total_conflicts", ["total_conflicts"]))
    if "fatalities" in df.columns:
        combos.append(("total_conflicts + fatalities", ["total_conflicts", "fatalities"]))

    results = []
    for label, extra_cols in combos:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = baseline_ppml_fe_inline(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception:
            pass

    return pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)


# --- 2) CSV preprocessing: log1p on conflict columns ---
def preprocess_csv_for_grid(csv_path):
    df = pd.read_csv(csv_path)
    cprefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in cprefixes)]
    if "fatalities" in df.columns:
        conflict_cols.append("fatalities")
    for col in conflict_cols:
        df[col] = np.log1p(df[col])
    return df


# --- 3) Define the grid ---
ROOT = Path("/Users/zenrehda/Desktop/bachelor_2026")

CSV_FILES = [
    # (filename, relative_path, source_type, lag, threshold)
    ("ecowas_df_full_lag_zero.csv",            "data/ecowas_df_full_lag_zero.csv",            "non-synthetic", 0, None),
    ("ecowas_df_full_lag_one.csv",             "data/ecowas_df_full_lag_one.csv",             "non-synthetic", 1, None),
    ("ecowas_df_full_lag_two.csv",             "data/ecowas_df_full_lag_two.csv",             "non-synthetic", 2, None),
    ("ecowas_df_synthetic_full_lag_zero_0.csv",   "data/synth/ecowas_df_synthetic_full_lag_zero_0.csv",   "synthetic", 0, 0),
    ("ecowas_df_synthetic_full_lag_zero_10.csv",  "data/synth/ecowas_df_synthetic_full_lag_zero_10.csv",  "synthetic", 0, 10),
    ("ecowas_df_synthetic_full_lag_zero_100.csv", "data/synth/ecowas_df_synthetic_full_lag_zero_100.csv", "synthetic", 0, 100),
    ("ecowas_df_synthetic_full_lag_one_0.csv",    "data/synth/ecowas_df_synthetic_full_lag_one_0.csv",    "synthetic", 1, 0),
    ("ecowas_df_synthetic_full_lag_one_10.csv",   "data/synth/ecowas_df_synthetic_full_lag_one_10.csv",   "synthetic", 1, 10),
    ("ecowas_df_synthetic_full_lag_one_100.csv",  "data/synth/ecowas_df_synthetic_full_lag_one_100.csv",  "synthetic", 1, 100),
    ("ecowas_df_synthetic_full_lag_two_0.csv",    "data/synth/ecowas_df_synthetic_full_lag_two_0.csv",    "synthetic", 2, 0),
    ("ecowas_df_synthetic_full_lag_two_10.csv",   "data/synth/ecowas_df_synthetic_full_lag_two_10.csv",   "synthetic", 2, 10),
    ("ecowas_df_synthetic_full_lag_two_100.csv",  "data/synth/ecowas_df_synthetic_full_lag_two_100.csv",  "synthetic", 2, 100),
]

TARGETS = ["tradeflow_baci", "tradeflow_imf_o", "tradeflow_imf_d",
           "tradeflow_comtrade_o", "tradeflow_comtrade_d", "combined_trade_baci"]

MODELS = {
    "XGBoost":       find_best_conflict_combo1,
    "Random Forest": find_best_conflict_combo2,
    "MLP":           find_best_conflict_combo3,
    "OLS Linear FE": find_best_conflict_combo4,
    "PPML FE":       find_best_conflict_combo5,
}

GRID_BASE_COLS = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

# --- 4) Output management with resumability ---
results_dir = ROOT / "results"
results_dir.mkdir(exist_ok=True)
out_csv = results_dir / "robustness_grid.csv"

if out_csv.exists():
    grid_results = pd.read_csv(out_csv)
    completed = set(zip(grid_results["source_csv"], grid_results["model"], grid_results["target"]))
    print(f"Resuming run: {len(grid_results)} rows loaded, {len(completed)} (csv,model,target) cells already complete.")
else:
    grid_results = pd.DataFrame()
    completed = set()
    print("Starting fresh run — no prior results found.")

total_cells     = len(CSV_FILES) * len(MODELS) * len(TARGETS)
done_cells      = len(completed)
remaining_cells = total_cells - done_cells

print(f"\nGrid plan: {len(CSV_FILES)} CSVs × {len(MODELS)} models × {len(TARGETS)} targets = {total_cells} cells")
print(f"Done: {done_cells}.  Remaining: {remaining_cells}.\n")

# --- 5) Main loop ---
t_start    = time.time()
csv_cache  = {}   # avoid re-reading the same CSV multiple times within this run

for csv_name, rel_path, source_type, lag, threshold in CSV_FILES:
    full_path = ROOT / rel_path
    if not full_path.exists():
        print(f"⚠ MISSING: {rel_path}")
        continue

    # Lazy load with caching
    if csv_name not in csv_cache:
        try:
            csv_cache[csv_name] = preprocess_csv_for_grid(full_path)
        except Exception as e:
            print(f"⚠ {csv_name} failed to load: {e}")
            continue
    df_csv = csv_cache[csv_name]

    for model_name, combo_fn in MODELS.items():
        for target in TARGETS:
            if target not in df_csv.columns:
                continue
            cell_key = (csv_name, model_name, target)
            if cell_key in completed:
                continue

            t0 = time.time()
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    combo_df = combo_fn(df_csv, target, base_columns=GRID_BASE_COLS)

                if len(combo_df) == 0:
                    print(f"  ⊘ {csv_name} | {model_name} | {target}  — no successful combos")
                    continue

                combo_df = combo_df.copy()
                combo_df["source_csv"]  = csv_name
                combo_df["source_type"] = source_type
                combo_df["lag"]         = lag
                combo_df["threshold"]   = threshold
                combo_df["model"]       = model_name
                combo_df["target"]      = target

                grid_results = pd.concat([grid_results, combo_df], ignore_index=True)
                grid_results.to_csv(out_csv, index=False)
                completed.add(cell_key)
                done_cells += 1

                elapsed       = time.time() - t0
                total_elapsed = time.time() - t_start
                rate          = done_cells / total_elapsed if total_elapsed > 0 else 0
                eta_min       = (total_cells - done_cells) / rate / 60 if rate > 0 else 0

                print(f"  [{done_cells:>3}/{total_cells}]  "
                      f"{csv_name:<48} | {model_name:<14} | {target:<22} "
                      f"({elapsed:>5.1f}s)  ETA: {eta_min:>5.0f} min")

            except Exception as e:
                print(f"  ⚠ {csv_name} | {model_name} | {target}  FAILED: {type(e).__name__}: {str(e)[:80]}")

print(f"\n✓ DONE. Total runtime: {(time.time() - t_start) / 60:.1f} min.")
print(f"  Final result row count: {len(grid_results)}")
print(f"  Saved to: {out_csv}")

In [ ]:
# === Step 13b: Summary pivots over the robustness grid ===
# Run after the grid completes. Loads results/robustness_grid.csv and produces
# (a) best model per (csv, target), (b) does conflict help anywhere?,
# (c) gravity-only R² heat-map by csv × target × model.

grid = pd.read_csv(ROOT / "results" / "robustness_grid.csv")
print(f"Loaded {len(grid):,} rows from robustness_grid.csv")
print(f"Unique combos: {grid[['source_csv','model','target']].drop_duplicates().shape[0]}")

# Helper: identify gravity-only rows (model_name describes conflict combo, gravity-only is "none" — but our
# combo functions don't produce a "none" row. Gravity-only exists only as a separate result outside the combo
# wrapper. So we approximate gravity-only as "the row with the smallest set of conflict cols" — which is
# "fatalities" (single col) or "total_conflicts" (single col). We'll just use the *worst* combo per (csv, model, target)
# as a rough lower bound and the *best* combo as the upper bound. For a true gravity-only baseline we would need
# to call xgboost_regression / etc. directly on each CSV, which is outside this grid's scope.)

# (a) Best conflict combo per (csv, model, target) — by R²
best_per_cell = (
    grid.sort_values("r2", ascending=False)
        .groupby(["source_csv", "model", "target"], as_index=False)
        .first()
)
print("\n=== BEST CONFLICT COMBO per (CSV × model × target), by R² ===")
print(f"  {len(best_per_cell)} cells")

# (b) For each (csv, target), which model wins?
winner_per_target = (
    best_per_cell.sort_values("r2", ascending=False)
                 .groupby(["source_csv", "target"], as_index=False)
                 .first()
                 .rename(columns={"model": "winner_model", "r2": "winner_r2"})
                 [["source_csv", "target", "winner_model", "model_name", "winner_r2"]]
)
print("\n=== WINNER MODEL per (CSV × target) ===")
winner_counts = winner_per_target["winner_model"].value_counts()
print(winner_counts.to_string())

# (c) Mean R² by model, averaged across all (csv, target) cells (using best conflict combo per cell)
mean_r2_by_model = best_per_cell.groupby("model")["r2"].agg(["mean", "median", "std", "count"]).round(4)
print("\n=== Mean R² by model (across all CSV × target cells, using best conflict combo) ===")
print(mean_r2_by_model.sort_values("mean", ascending=False).to_string())

# (d) Mean R² by CSV (averaged across model × target × best combo)
mean_r2_by_csv = best_per_cell.groupby(["source_type", "lag", "threshold", "source_csv"])["r2"].agg(["mean", "count"]).round(4)
print("\n=== Mean R² by CSV (averaged across model × target) ===")
print(mean_r2_by_csv.sort_values("mean", ascending=False).to_string())

# (e) Mean R² by target (averaged across CSV × model × best combo)
mean_r2_by_target = best_per_cell.groupby("target")["r2"].agg(["mean", "median", "count"]).round(4)
print("\n=== Mean R² by target (averaged across CSV × model) ===")
print(mean_r2_by_target.sort_values("mean", ascending=False).to_string())

# (f) Best (csv, model, target, conflict_combo) overall
top10 = best_per_cell.nlargest(10, "r2")[["source_csv", "model", "target", "model_name", "rmse", "mae", "r2"]]
print("\n=== TOP 10 specifications across the entire grid ===")
print(top10.to_string(index=False))

# (g) Bottom 10 (worst-fitting cells)
bot10 = best_per_cell.nsmallest(10, "r2")[["source_csv", "model", "target", "model_name", "rmse", "mae", "r2"]]
print("\n=== BOTTOM 10 specifications ===")
print(bot10.to_string(index=False))

# Save the pivots
(ROOT / "results" / "robustness_winner_per_target.csv").write_text(winner_per_target.to_csv(index=False))
(ROOT / "results" / "robustness_best_per_cell.csv").write_text(best_per_cell.to_csv(index=False))
print(f"\nSummary tables saved to {ROOT / 'results'}/")